In [19]:
import os
import numpy as np
import pandas as pd
import torch
from darts.models import TFTModel

from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
import math 
from dateutil.relativedelta import relativedelta
from snowflake_utils import *
from darts import TimeSeries
from darts.dataprocessing.transformers import (
    Scaler,
    StaticCovariatesTransformer,
)

In [8]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

In [9]:
WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Modelling\darts_logs"

CKPT_MODEL_NAME = "daily_festive_tft_boss_approach"

print("Loading best checkpoint...")
best_model = TFTModel.load_from_checkpoint(
    CKPT_MODEL_NAME, work_dir=WORK_DIR, best=True
)
best_model.model.eval()
print(f"  loaded: {CKPT_MODEL_NAME}")
print(f"  input_chunk_length  : {best_model.input_chunk_length}")
print(f"  output_chunk_length : {best_model.output_chunk_length}")


Loading best checkpoint...
  loaded: daily_festive_tft_boss_approach
  input_chunk_length  : 122
  output_chunk_length : 122


In [10]:
#Read the table
df = session.sql("""SELECT * FROM 
                 MOP_DATABASE.SOQ.DAILY_FORECASTING_DATA_FOR_MODELLING_TFT
                 WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (SELECT PARENT_DEALER_CODE_MODEL_FAMILY
                FROM MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING)""")

In [14]:
target_col = "NET_SALES"
group_col = "PARENT_DEALER_CODE_MODEL_FAMILY"
time_col = "SERIES_INDEX"


#Static covariates
static_covariates = [
    "PARENT_DEALER_CODE",
    "MODEL_FAMILY",
    "MODEL_FAMILY_CODE",
    "MODEL_NAME",
    "BRAKE_TYPE",
    "IGNITION_TYPE",
    "WHEEL_TYPE",
    "COLOUR",
    "DEALER_CITY",
    "X_CITY_CATEGORY",
    "ZONAL_OFFICE_NAME",
]


#Known future covariates
calendar_covariates = [
    "YEAR",
    "MONTH",
    "DAY_OF_THE_WEEK",
    "DAY_OF_THE_MONTH",
]

festival_covariates = [
    "HARTALIK_TEEJ",
    "GANESH_CHATURTHI",
    "JANMASHTAMI",
    "VISHWAKARMA_PUJA",
    "KARWA_CHAUTH",
    "ONAM",
    "MARRIAGE_DAY",

    "N-16", "N-15", "N-14", "N-13",
    "N-12", "N-11", "N-10", "N-9",
    "N-8", "N-7", "N-6", "N-5",
    "N-4", "N-3", "N-2", "N-1",
    "N",
    "N+1", "N+2", "N+3", "N+4",
    "N+5", "N+6", "N+7", "N+8", "N+9",

    "D-3", "D-2", "D-1", "D",
    "D+1", "D+2", "D+3",
    "D+4", "D+5", "D+6",
]

future_covariates = calendar_covariates + festival_covariates

In [11]:
df = df.drop('DATASET_TYPE')
data = df.to_pandas()

In [12]:
data = data.sort_values(
    [
        "PARENT_DEALER_CODE_MODEL_FAMILY",
        "SERIES_INDEX",
    ]
).reset_index(drop=True)

In [13]:
weekday_mapping = {
    "MON": 0,
    "TUE": 1,
    "WED": 2,
    "THU": 3,
    "FRI": 4,
    "SAT": 5,
    "SUN": 6,
}

if data["DAY_OF_THE_WEEK"].dtype == "object":
    data["DAY_OF_THE_WEEK"] = (
        data["DAY_OF_THE_WEEK"]
        .str.upper()
        .map(weekday_mapping)
    )

In [15]:

if data[future_covariates].isna().any().any():
    null_counts = (
        data[future_covariates]
        .isna()
        .sum()
    )

    print(
        null_counts[null_counts > 0]
        .sort_values(ascending=False)
    )

    raise ValueError(
        "Null values found in known future covariates."
    )

In [16]:
historical_data = data[
    data["YEAR"].isin([2023, 2024, 2025])
].copy()

future_2026_data = data[
    data["YEAR"].eq(2026)
].copy()


In [17]:
if historical_data[target_col].isna().any():
    raise ValueError(
        "Historical NET_SALES contains missing values."
    )

if future_2026_data[target_col].notna().any():
    print(
        "Warning: Some 2026 NET_SALES values are populated. "
        "Ensure these are not placeholder zeroes."
    )


target_input_data = historical_data[
    [time_col, group_col, target_col]
    + static_covariates
].copy()


In [20]:

target_series = TimeSeries.from_group_dataframe(
    df=target_input_data,
    group_cols=group_col,
    time_col=time_col,
    value_cols=target_col,
    static_cols=static_covariates,
    freq=1,
    fill_missing_dates=False,
)

print("Number of series:", len(target_series))
print("Length of first series:", len(target_series[0]))
print("First series index:", target_series[0].time_index[:5])
print("Last series index:", target_series[0].time_index[-5:])
print("Static covariates:")
print(target_series[0].static_covariates)

training_covariate_data = historical_data[
    [time_col, group_col]
    + future_covariates
].copy()

all_covariate_data = data[
    [time_col, group_col]
    + future_covariates
].copy()

training_future_covariates = (
    TimeSeries.from_group_dataframe(
        df=training_covariate_data,
        group_cols=group_col,
        time_col=time_col,
        value_cols=future_covariates,
        freq=1,
        fill_missing_dates=False,
    )
)

full_future_covariates = (
    TimeSeries.from_group_dataframe(
        df=all_covariate_data,
        group_cols=group_col,
        time_col=time_col,
        value_cols=future_covariates,
        freq=1,
        fill_missing_dates=False,
    )
)

assert len(target_series) == len(training_future_covariates)
assert len(target_series) == len(full_future_covariates)

print(len(training_future_covariates[0]))  # 366
print(len(full_future_covariates[0]))      # 488


Number of series: 18080
Length of first series: 366
First series index: RangeIndex(start=1, stop=6, step=1, name='SERIES_INDEX')
Last series index: RangeIndex(start=362, stop=367, step=1, name='SERIES_INDEX')
Static covariates:
static_covariates          PARENT_DEALER_CODE_MODEL_FAMILY PARENT_DEALER_CODE  \
NET_SALES          10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE              10001   

static_covariates MODEL_FAMILY                 MODEL_FAMILY_CODE MODEL_NAME  \
NET_SALES              DESTINI  DESTINI<>DRUM<>SELF<>CAST<>WHITE    DESTINI   

static_covariates BRAKE_TYPE IGNITION_TYPE WHEEL_TYPE COLOUR DEALER_CITY  \
NET_SALES               DRUM          SELF       CAST  WHITE    AMRITSAR   

static_covariates X_CITY_CATEGORY     ZONAL_OFFICE_NAME  
NET_SALES                   URBAN  Zonal Office - North  
366
488


In [21]:
validation_start_index = 244

train_series = []
validation_series = []

for series in target_series:
    train_part = series.slice(
        series.start_time(),
        245,
    )

    validation_part = series.slice(
        123,
        series.end_time()+1,
    )

    train_series.append(train_part)
    validation_series.append(validation_part)


train_covariates = []
validation_covariates = []

for covariate_series in training_future_covariates:
    train_covariates.append(
        covariate_series.slice(
            covariate_series.start_time(),
            245,
        )
    )

    validation_covariates.append(
        covariate_series
    )

In [22]:

target_scaler = Scaler(
    global_fit=False,
    n_jobs=-1,
)

future_covariate_scaler = Scaler(
    global_fit=True,
    n_jobs=-1,
)

static_transformer = StaticCovariatesTransformer(
    n_jobs=-1,
)


scaled_train_series = (
    target_scaler.fit_transform(train_series)
)

print(f"Scaled_train_series_1 success!")

scaled_train_series = (
    static_transformer.fit_transform(
        scaled_train_series
    )
)

print(f"Scaled_train_series_2 success!")

scaled_validation_series = (
    target_scaler.transform(validation_series)
)

print(f"Scaled_validation_series_1 success!")


scaled_validation_series = (
    static_transformer.transform(
        scaled_validation_series
    )
)

print(f"Scaled_validation_series_2 success!")


scaled_train_covariates = (
    future_covariate_scaler.fit_transform(
        train_covariates
    )
)

print(f"scaled_train_covariates success!")


scaled_validation_covariates = (
    future_covariate_scaler.transform(
        validation_covariates
    )
)

print(f"scaled_validation_covariates success!")

scaled_full_covariates = future_covariate_scaler.transform(full_future_covariates)
print("scaled_full_covariates success!")
print(f"  length: {len(scaled_full_covariates[0])}  "
      f"(index {scaled_full_covariates[0].start_time()}"
      f"..{scaled_full_covariates[0].end_time()})")

# ---- NEW: also scale the full target series, for predicting from 2025 ----
scaled_full_series = target_scaler.transform(target_series)
scaled_full_series = static_transformer.transform(scaled_full_series)
print(f"scaled_full_series success! length: {len(scaled_full_series[0])}")

Scaled_train_series_1 success!
Scaled_train_series_2 success!
Scaled_validation_series_1 success!
Scaled_validation_series_2 success!
scaled_train_covariates success!
scaled_validation_covariates success!
scaled_full_covariates success!
  length: 488  (index 1..488)
scaled_full_series success! length: 366


In [23]:
print("\nPreparing inputs...")
print(f"  target_series          : {len(target_series)} series, "
      f"len={len(target_series[0])}")
print(f"  full_future_covariates : {len(full_future_covariates)} series, "
      f"len={len(full_future_covariates[0])}")

assert len(target_series) == len(full_future_covariates), \
    "series count mismatch between targets and covariates"

scaled_full_series = target_scaler.transform(target_series)
scaled_full_series = static_transformer.transform(scaled_full_series)

scaled_full_covariates = future_covariate_scaler.transform(full_future_covariates)


Preparing inputs...
  target_series          : 18080 series, len=366
  full_future_covariates : 18080 series, len=488


In [27]:
# Match the dtype used during training
scaled_full_series     = [s.astype(np.float32) for s in scaled_full_series]
scaled_full_covariates = [c.astype(np.float32) for c in scaled_full_covariates]

In [29]:
print(scaled_full_covariates[0].dtype)

float32


In [31]:
s = scaled_full_series[0]
c = scaled_full_covariates[0]

print("series values   :", s.dtype)
print("series statics  :", s.static_covariates.dtypes.unique() if s.static_covariates is not None else None)
print("covariate values:", c.dtype)
print("model params    :", next(best_model.model.parameters()).dtype)
print("precision       :", best_model.trainer_params.get("precision"))

# any series in the lists still float64?
bad_s = [i for i, x in enumerate(scaled_full_series) if x.dtype != np.float32]
bad_c = [i for i, x in enumerate(scaled_full_covariates) if x.dtype != np.float32]
print("float64 series   :", len(bad_s))
print("float64 covariates:", len(bad_c))


series values   : float32
series statics  : [dtype('float32')]
covariate values: float32
model params    : torch.float32
precision       : bf16-mixed
float64 series   : 0
float64 covariates: 0


In [ ]:
N_STEPS = 122      # 2026 window length

print(f"\nPredicting {N_STEPS} steps ahead for {len(scaled_full_series)} series...")

# best_model.trainer_params = {
#     "accelerator" : "gpu" if torch.cuda.is_available() else "cpu",
#     "devices" : 1
# }

with torch.no_grad():
    scaled_2026 = best_model.predict(
        n=N_STEPS,
        series=scaled_full_series,
        future_covariates=scaled_full_covariates,
        verbose=True,
    )

print(f"Forecast index range: {scaled_2026[0].start_time()} → {scaled_2026[0].end_time()}")

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Predicting 122 steps ahead for 18080 series...


c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

RuntimeError: expected mat1 and mat2 to have the same dtype, but got: double != float